In [1]:
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34
from torch.utils.tensorboard import SummaryWriter
import shutup
import segmentation_models_pytorch as smp
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()
print(1)

1


In [2]:
class TrainDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform, crop=512):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.crop = crop
        
        self.images = sorted([f for f in os.listdir(images_dir) if f.endswith(('.tiff'))])
        self.masks = sorted([f for f in os.listdir(masks_dir) if f.endswith(('.tif'))])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])

        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path)

        w, h = image.size
        new_h = random.randint(0, h - self.crop)
        new_w = random.randint(0, w - self.crop)

        image = image.crop((new_w, new_h, new_w + self.crop, new_h + self.crop))
        mask = mask.crop((new_w, new_h, new_w + self.crop, new_h + self.crop))

        augmented = self.transform(image=np.array(image), mask=np.array(mask))
        image = augmented['image']
        mask = augmented['mask'] 
            
        mask = (mask > 0).float().unsqueeze(0)
        return image, mask

class TestDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform, crop=512):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.crop = crop
        self.crop_coords = [0, 494, 988]
        
        self.images = sorted([f for f in os.listdir(images_dir) if f.endswith(('.tiff'))])
        self.masks = sorted([f for f in os.listdir(masks_dir) if f.endswith(('.tif'))])
    
    def __len__(self):
        return len(self.images) * 9
    
    def __getitem__(self, idx):
        file_idx = idx // 9
        patch_idx = idx % 9
        row = patch_idx // 3
        col = patch_idx % 3
        new_h = self.crop_coords[row]
        new_w = self.crop_coords[col]
        
        img_path = os.path.join(self.images_dir, self.images[file_idx])
        mask_path = os.path.join(self.masks_dir, self.masks[file_idx])
        
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path)
        
        image = image.crop((new_w, new_h, new_w + self.crop, new_h + self.crop))
        mask = mask.crop((new_w, new_h, new_w + self.crop, new_h + self.crop))
        
        augmented = self.transform(image=np.array(image), mask=np.array(mask))
        image = augmented['image']
        mask = augmented['mask'] 
        
        mask = (mask > 0).float().unsqueeze(0)
        return image, mask, new_h, new_w, file_idx

<h16>Загрузка, преобразование и разделение датасета на train, валидацию и test<h16>

In [3]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_dataset = TrainDataset(images_dir='./archive/tiff/train', masks_dir='./archive/tiff/train_labels', transform=train_transform)
valid_dataset = TestDataset(images_dir='./archive/tiff/val', masks_dir='./archive/tiff/val_labels', transform=test_transform)
test_dataset = TestDataset(images_dir='./archive/tiff/test', masks_dir='./archive/tiff/test_labels', transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [4]:
class SegmRoadModel(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        
        resnet = resnet34(pretrained=True)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.dec4 = self.decode(512, 256)
        self.dec3 = self.decode(512, 128)
        self.dec2 = self.decode(256, 64)
        self.dec1 = self.decode(128, 32)
        self.final = nn.Conv2d(32, 1, kernel_size=1)

    def decode(self, inC, outC, out=False):
        layer = [
            nn.ConvTranspose2d(inC, outC, 2, stride=2),
            nn.Conv2d(outC, outC, 3, padding=1),
            nn.BatchNorm2d(outC),
            nn.ReLU(inplace=True)
        ]
        
        if not out:
            layer.extend([
                nn.Conv2d(outC, outC, 3, padding=1),
                nn.BatchNorm2d(outC),
                nn.ReLU(inplace=True)
            ])
        return nn.Sequential(*layer)
        
    
    def forward(self, x):
        e0 = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        e1 = self.layer1(e0) 
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        
        d4 = self.dec4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d1 = self.dec1(d2)
        out = self.final(d1)
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        return out

In [5]:
def iou_score(pred, target, eps=1e-6):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + eps) / (union + eps)

bce_loss  = smp.losses.SoftBCEWithLogitsLoss(pos_weight=torch.tensor(21.10).cuda())
dice_loss = smp.losses.DiceLoss(mode='binary')

def comb_loss(preds, targets):
    return 0.7 * dice_loss(preds, targets) + 0.3 * bce_loss(preds, targets)

In [6]:
def train_model(model, train_loader, valid_loader, optimizer, loss_func, num_epochs, output, curr_iou):
    best_valid_iou = curr_iou
    for epoch in range(num_epochs):
        print(f'\nЭпоха {epoch+1}/{num_epochs}')

        # Обучение
        model.train()
        train_loss = 0
        train_iou = 0

        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()

            optimizer.zero_grad()
            logits = model(images)
            loss = loss_func(logits, labels)
            loss.backward()
            optimizer.step()

            iou = iou_score(logits, labels)
            train_iou += iou.item()
            train_loss += loss.item()

        # Валидация
        model.eval()
        valid_loss = 0
        valid_iou = 0

        with torch.no_grad():
            for images, labels, _, _, _ in valid_loader:
                images, labels = images.cuda(), labels.cuda()
                logits = model(images)
                loss = loss_func(logits, labels)

                iou = iou_score(logits, labels)
                valid_iou += iou.item()
                valid_loss += loss.item()

        train_loss /= len(train_loader)
        train_iou /= len(train_loader)
        valid_loss /= len(valid_loader)
        valid_iou /= len(valid_loader)

        output.add_scalars('Loss', {'train': train_loss, 'valid': valid_loss}, epoch)
        output.add_scalars('IoU', {'train': train_iou, 'valid': valid_iou}, epoch)

        print(f'Train Loss: {train_loss:.4f}, Train IoU: {train_iou:.4f}')
        print(f'Valid Loss: {valid_loss:.4f}, Valid IoU: {valid_iou:.4f}')

        if valid_iou > best_valid_iou:
            best_valid_iou = valid_iou
            torch.save(model.state_dict(), 'best_model_4.pth')

In [7]:
def test_model(model, test_loader, loss_func):
    print("Test")

    model.load_state_dict(torch.load('best_model_4.pth'))
    model.eval()

    test_loss = 0
    test_iou = 0
    pred_maps = {}
    count_maps = {}
    num_images = len(test_loader.dataset.images)
    
    with torch.no_grad():
        for images, masks, heights, widths, idxs in test_loader:
            images, masks = images.cuda(), masks.cuda()
            
            outputs = model(images)
            loss = loss_func(outputs, masks)
            test_loss += loss.item()
            
            probs = torch.sigmoid(outputs).cpu()
            masks_cpu = masks.cpu()
            
            for i in range(len(images)):
                idx = idxs[i].item()
                h = heights[i].item()
                w = widths[i].item()
                
                if idx not in pred_maps:
                    pred_maps[idx] = torch.zeros((1, 1500, 1500), dtype=torch.float32)
                    count_maps[idx] = torch.zeros((1, 1500, 1500), dtype=torch.float32)
                
                pred_maps[idx][:, h:h+512, w:w+512] += probs[i]
                count_maps[idx][:, h:h+512, w:w+512] += 1
    
    for idx in pred_maps:
        final_probs = pred_maps[idx] / count_maps[idx]
        final_pred = (final_probs > 0.5).float()
        
        mask_path = test_loader.dataset.masks[idx]
        full_mask = Image.open(f'./archive/tiff/test_labels/{mask_path}')
        full_mask_tensor = transforms.ToTensor()(full_mask).unsqueeze(0).float()
        full_mask_tensor = (full_mask_tensor > 0).float()
        
        test_iou += iou_score(final_pred, full_mask_tensor)
    
    test_loss /= len(test_loader)
    test_iou /= num_images
    print(f'Test Loss: {test_loss:.4f}, Test IoU: {test_iou:.4f}')

In [8]:
shutup.please()

model = SegmRoadModel().cuda()
output = SummaryWriter('log_part_4')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    loss_func=comb_loss,
    num_epochs=100,
    output=output,
    curr_iou=0
)
output.close()


Эпоха 1/100
Train Loss: 0.7987, Train IoU: 0.2118
Valid Loss: 0.6699, Valid IoU: 0.3139

Эпоха 2/100
Train Loss: 0.6875, Train IoU: 0.3243
Valid Loss: 0.6054, Valid IoU: 0.3390

Эпоха 3/100
Train Loss: 0.6663, Train IoU: 0.3506
Valid Loss: 0.5962, Valid IoU: 0.4913

Эпоха 4/100
Train Loss: 0.6501, Train IoU: 0.3603
Valid Loss: 0.5604, Valid IoU: 0.4216

Эпоха 5/100
Train Loss: 0.6414, Train IoU: 0.3755
Valid Loss: 0.5240, Valid IoU: 0.4614

Эпоха 6/100
Train Loss: 0.6115, Train IoU: 0.4064
Valid Loss: 0.5279, Valid IoU: 0.4129

Эпоха 7/100
Train Loss: 0.6150, Train IoU: 0.4000
Valid Loss: 0.4968, Valid IoU: 0.4753

Эпоха 8/100
Train Loss: 0.6015, Train IoU: 0.4118
Valid Loss: 0.5097, Valid IoU: 0.4658

Эпоха 9/100
Train Loss: 0.5942, Train IoU: 0.4203
Valid Loss: 0.4916, Valid IoU: 0.4645

Эпоха 10/100
Train Loss: 0.6084, Train IoU: 0.4123
Valid Loss: 0.5273, Valid IoU: 0.4946

Эпоха 11/100
Train Loss: 0.5966, Train IoU: 0.4231
Valid Loss: 0.4960, Valid IoU: 0.5196

Эпоха 12/100
Train

In [11]:
%load_ext tensorboard
%tensorboard --logdir log_part_4

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [10]:
shutup.please()
test_model(model=model, test_loader=test_loader, loss_func=comb_loss)

Test
Test Loss: 0.3576, Test IoU: 0.6016
